# Noetia — Whisper Transcription Pipeline

Runs on a free GPU (T4) in Google Colab. Transcribes a LibriVox audiobook with word-level timestamps and produces a merged `.vtt` file ready for `seed-sync-whisper.js`.

**Steps:**
1. Run Cell 1 — check GPU
2. Run Cell 2 — install dependencies
3. Edit Cell 3 — set the book URL and slug
4. Run Cell 4 — load pipeline functions
5. Run Cell 5 — mount Google Drive (saves progress across disconnects)
6. Run Cell 6 — run the full pipeline
7. Run Cell 7 — download the merged VTT

> **Important:** Go to `Runtime → Change runtime type → T4 GPU → Save` before starting.
>
> **If the session disconnects mid-way:** re-run Cells 1–5, then Cell 6. Already-transcribed chapters are saved to Drive and will be skipped automatically.

In [ ]:
# CELL 1 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU found!')
    for line in result.stdout.split('\n')[:12]:
        print(line)
else:
    print('NO GPU FOUND.')
    print('Go to: Runtime → Change runtime type → T4 GPU → Save, then re-run.')

In [ ]:
# CELL 2 — Install dependencies
!pip install -q openai-whisper
!apt-get install -q -y ffmpeg
print('Dependencies ready.')

In [ ]:
# CELL 3 — Configuration
# Edit these values for each new book

LIBRIVOX_URL = 'https://librivox.org/don-quijote-vol-1-by-miguel-de-cervantes-saavedra/'
SLUG        = 'don-quijote-vol-1'   # output filename: <slug>.merged.vtt
BOOK_DIR    = 'Don Quijote Vol. I'  # subfolder name for chapter VTTs
LANGUAGE    = 'es'
MODEL       = 'medium'              # medium = fast + accurate on GPU; large = max accuracy

print(f'Book  : {BOOK_DIR}')
print(f'URL   : {LIBRIVOX_URL}')
print(f'Slug  : {SLUG}')
print(f'Model : {MODEL}  |  Language: {LANGUAGE}')

In [ ]:
# CELL 4 — Pipeline functions (no edits needed)
import json
import re
import subprocess
import urllib.parse
import urllib.request
from pathlib import Path

# Paths — overridden in Cell 5 after Drive is mounted
AUDIO_DIR  = Path('/content/audio') / SLUG
VTT_DIR    = Path('/content/transcriptions') / BOOK_DIR
MERGED_OUT = Path('/content/transcriptions') / f'{SLUG}.merged.vtt'
GAP_SECONDS = 2.0

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
VTT_DIR.mkdir(parents=True, exist_ok=True)


def fetch(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 Noetia-Whisper-Bot'})
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read().decode('utf-8', errors='replace')


def fetch_bytes(url, dest, label=''):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 Noetia-Whisper-Bot'})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, 'wb') as f:
        total = int(r.headers.get('Content-Length', 0))
        downloaded = 0
        while chunk := r.read(65536):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded * 100 // total
                print(f'\r  {label}: {pct}%', end='', flush=True)
    print()


def scrape_librivox(librivox_url):
    print(f'Fetching: {librivox_url}')
    html = fetch(librivox_url)
    m = re.search(r'href="(https://archive\.org/compress/([^"]+)\.zip[^"]*)"', html, re.I)
    if m:
        identifier = re.search(r'archive\.org/(?:compress|download)/([^/?#]+)', m.group(1)).group(1)
        print(f'  Identifier (zip): {identifier}')
        return identifier
    m = re.search(r'href="https://archive\.org/(?:compress|download)/([^/"]+)', html, re.I)
    if m:
        print(f'  Identifier (generic): {m.group(1)}')
        return m.group(1)
    raise RuntimeError('Could not find archive.org identifier on LibriVox page')


def get_chapter_mp3s(identifier):
    api_url = f'https://archive.org/metadata/{identifier}'
    print(f'Querying metadata: {api_url}')
    meta = json.loads(fetch(api_url))
    files = meta.get('files', [])
    mp3s = [f for f in files if f.get('name', '').lower().endswith('.mp3') and
            f.get('format', '').lower() in ('mp3', 'vbr mp3', '')]
    kb64 = [f for f in mp3s if '64kb' in f.get('name', '').lower() or '64kb' in f.get('format', '').lower()]
    chosen = kb64 if kb64 else mp3s
    if not chosen:
        raise RuntimeError(f'No MP3 files found for: {identifier}')
    chosen.sort(key=lambda f: tuple(int(n) for n in re.findall(r'\d+', f.get('name', ''))))
    print(f'  Found {len(chosen)} chapter MP3s')
    return chosen, identifier


def download_chapters(identifier, files):
    base = f'https://archive.org/download/{identifier}'
    paths = []
    for i, f in enumerate(files):
        name = f['name']
        local = AUDIO_DIR / name
        if local.exists():
            print(f'  [{i+1}/{len(files)}] Already downloaded: {name}')
        else:
            print(f'  [{i+1}/{len(files)}] Downloading: {name}')
            fetch_bytes(f'{base}/{urllib.parse.quote(name)}', local, label=name)
        paths.append(local)
    return paths


def run_whisper(audio_path):
    vtt = VTT_DIR / (audio_path.stem + '.vtt')
    if vtt.exists():
        print(f'  Already done: {audio_path.name}')
        return vtt
    cmd = [
        'whisper', str(audio_path),
        '--model', MODEL,
        '--language', LANGUAGE,
        '--word_timestamps', 'True',
        '--output_format', 'vtt',
        '--output_dir', str(VTT_DIR),
    ]
    print(f'  Transcribing: {audio_path.name} ...', end='', flush=True)
    result = subprocess.run(cmd, capture_output=True, text=True)
    combined = result.stdout + result.stderr
    if result.returncode != 0 or 'Skipping' in combined:
        print(f'\n  FAILED:')
        print(combined[-2000:])
        if 'ffmpeg' in combined:
            raise RuntimeError('ffmpeg not found — re-run Cell 2')
        raise RuntimeError(f'Whisper failed on {audio_path.name}')
    print(' done')
    if not vtt.exists():
        vtts = list(VTT_DIR.glob('*.vtt'))
        raise RuntimeError(f'VTT not found. Files in dir: {[v.name for v in vtts]}')
    return vtt


def ts_to_sec(ts):
    parts = ts.strip().replace(',', '.').split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    return int(parts[0]) * 60 + float(parts[1])


def sec_to_ts(s):
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = s % 60
    return f'{h:02d}:{m:02d}:{sec:06.3f}'


def shift_inline(text, offset):
    def replace_ts(m):
        return f'<{sec_to_ts(ts_to_sec(m.group(1)) + offset)}>'
    return re.sub(r'<(\d{2}:\d{2}:\d{2}[\.,]\d{3})>', replace_ts, text)


def parse_cues(vtt):
    cues = []
    for block in re.split(r'\n\n+', vtt.strip()):
        lines = block.strip().splitlines()
        for i, line in enumerate(lines):
            m = re.match(r'(\d{2}:\d{2}:\d{2}[\.,]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[\.,]\d{3})', line)
            if m:
                cues.append((ts_to_sec(m.group(1)), ts_to_sec(m.group(2)), '\n'.join(lines[i+1:])))
                break
    return cues


def merge_vtts():
    files = sorted(VTT_DIR.glob('*.vtt'), key=lambda f: tuple(int(n) for n in re.findall(r'\d+', f.name)))
    if not files:
        raise RuntimeError(f'No VTT files found in {VTT_DIR}')
    all_cues = []
    offset = 0.0
    for path in files:
        cues = parse_cues(path.read_text(encoding='utf-8'))
        if not cues:
            continue
        for start, end, payload in cues:
            all_cues.append((start + offset, end + offset, shift_inline(payload, offset)))
        last_end = max(end for _, end, _ in cues)
        offset += last_end + GAP_SECONDS
        print(f'  + {path.name} ({len(cues)} cues)')
    lines = ['WEBVTT', '']
    for i, (start, end, payload) in enumerate(all_cues, 1):
        lines += [str(i), f'{sec_to_ts(start)} --> {sec_to_ts(end)}', payload, '']
    MERGED_OUT.write_text('\n'.join(lines), encoding='utf-8')
    print(f'\nMerged {len(all_cues)} cues → {MERGED_OUT.name}')


print('Functions loaded.')

In [ ]:
# CELL 5 — Mount Google Drive
# VTTs are saved here so they survive session disconnects.
# On reconnect: re-run Cells 1-5, then Cell 6 skips already-done chapters.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = Path('/content/drive/MyDrive/noetia-whisper')
VTT_DIR    = DRIVE_DIR / BOOK_DIR
MERGED_OUT = DRIVE_DIR / f'{SLUG}.merged.vtt'
VTT_DIR.mkdir(parents=True, exist_ok=True)

print(f'VTTs will be saved to: {VTT_DIR}')
print(f'Merged output will be: {MERGED_OUT}')
already_done = list(VTT_DIR.glob('*.vtt'))
print(f'Already transcribed: {len(already_done)} chapters')

In [ ]:
# CELL 6 — Run the pipeline
# Downloads chapters, transcribes, and merges.
# Already-done chapters (saved to Drive) are skipped automatically.
# If disconnected mid-way: re-run Cells 1-5, then re-run this cell.

if MERGED_OUT.exists():
    print(f'Merged VTT already exists: {MERGED_OUT}')
    print('Delete it from Drive and re-run to regenerate.')
else:
    identifier = scrape_librivox(LIBRIVOX_URL)
    files, identifier = get_chapter_mp3s(identifier)

    print(f'\nDownloading {len(files)} chapters...')
    audio_paths = download_chapters(identifier, files)

    print(f'\nTranscribing {len(audio_paths)} chapters with {MODEL} model...')
    for i, audio in enumerate(audio_paths):
        print(f'[{i+1}/{len(audio_paths)}]', end=' ')
        run_whisper(audio)

    print('\nMerging VTTs...')
    merge_vtts()

    print(f'\nDone! File saved to Drive: {MERGED_OUT}')

In [ ]:
# CELL 7 — Download the merged VTT to your computer
from google.colab import files as colab_files

if MERGED_OUT.exists():
    print(f'Downloading: {MERGED_OUT.name}')
    colab_files.download(str(MERGED_OUT))
else:
    print('Merged VTT not found. Run Cell 6 first.')
    print(f'Expected at: {MERGED_OUT}')